<a href="https://colab.research.google.com/github/rajneesh1846-lang/CareLoop-NoShow-Guard/blob/main/03_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
print("Total no-shows:", df["no_show"].sum())
print(df.head())


Saving appointments_clean.csv to appointments_clean.csv
Dataset shape: (990, 16)
Total no-shows: 218
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

y = df["no_show"]

print("Number of features:", X.shape[1])
print("Feature names:", X.columns.tolist())


Number of features: 11
Feature names: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Test no-shows:", y_test.sum())


Training rows: 792
Testing rows: 198
Test no-shows: 49


In [ ]:

baseline_predictions = np.zeros(len(y_test), dtype=int)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline accuracy:", round(baseline_accuracy, 4))


Baseline accuracy: 0.7525


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Training shape:", X_train_s.shape)
print("Testing shape:", X_test_s.shape)


Training shape: (792, 11)
Testing shape: (198, 11)


In [ ]:

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_s, y_train)

lr_predictions = lr.predict(X_test_s)
lr_probabilities = lr.predict_proba(X_test_s)[:, 1]

print("Logistic Regression model trained successfully!")


Logistic Regression model trained successfully!


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:",
      round(accuracy_score(y_test, lr_predictions), 3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))

print("Precision:",
      round(precision_score(y_test, lr_predictions), 3))

print("Recall:",
      round(recall_score(y_test, lr_predictions), 3))

print("F1 Score:",
      round(f1_score(y_test, lr_predictions), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, lr_probabilities), 3))


Accuracy: 0.788
Confusion Matrix:
[[139  10]
 [ 32  17]]
Precision: 0.63
Recall: 0.347
F1 Score: 0.447
ROC-AUC: 0.789


In [ ]:

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_[0]
})

print(coefficients.to_string(index=False))

reminder_index = X.columns.get_loc("reminder_sent")
reminder_coef = lr.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

print("\nReminder coefficient:", reminder_coef)

print("Odds ratio per 1 standard deviation:",
      np.exp(reminder_coef))

print("Reminder standard deviation:", reminder_scale)

print("Odds ratio for No to Yes:",
      np.exp(reminder_coef / reminder_scale))


                Feature  Coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658

Reminder coefficient: -0.5806686033659922
Odds ratio per 1 standard deviation: 0.559524141750829
Reminder standard deviation: 0.4949752621616814
Odds ratio for No to Yes: 0.3093980903124433


In [ ]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

full_tree = DecisionTreeClassifier(
    random_state=42
)

full_tree.fit(X_train, y_train)

full_train_accuracy = accuracy_score(
    y_train, full_tree.predict(X_train)
)

full_test_accuracy = accuracy_score(
    y_test, full_tree.predict(X_test)
)

print("Full Tree Depth:", full_tree.get_depth())
print("Training Accuracy:", round(full_train_accuracy, 3))
print("Testing Accuracy:", round(full_test_accuracy, 3))


Full Tree Depth: 15
Training Accuracy: 1.0
Testing Accuracy: 0.672


In [ ]:

depth3_tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

depth3_tree.fit(X_train, y_train)

depth3_train_accuracy = accuracy_score(
    y_train, depth3_tree.predict(X_train)
)

depth3_test_accuracy = accuracy_score(
    y_test, depth3_tree.predict(X_test)
)

print("Depth-3 Tree Depth:", depth3_tree.get_depth())
print("Training Accuracy:", round(depth3_train_accuracy, 3))
print("Testing Accuracy:", round(depth3_test_accuracy, 3))


Depth-3 Tree Depth: 3
Training Accuracy: 0.822
Testing Accuracy: 0.778


In [ ]:

tree_comparison = pd.DataFrame({
    "Model": ["Full Decision Tree", "Depth-3 Tree"],
    "Training Accuracy": [
        full_train_accuracy,
        depth3_train_accuracy
    ],
    "Testing Accuracy": [
        full_test_accuracy,
        depth3_test_accuracy
    ]
})

tree_comparison["Accuracy Gap"] = (
    tree_comparison["Training Accuracy"]
    - tree_comparison["Testing Accuracy"]
)

print(tree_comparison.round(3).to_string(index=False))


             Model  Training Accuracy  Testing Accuracy  Accuracy Gap
Full Decision Tree              1.000             0.672         0.328
      Depth-3 Tree              0.822             0.778         0.044


In [ ]:

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_predictions = rf.predict(X_test)
rf_probabilities = rf.predict_proba(X_test)[:, 1]

print("Random Forest trained successfully!")


Random Forest trained successfully!


In [ ]:

rf_train_accuracy = accuracy_score(
    y_train, rf.predict(X_train)
)

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_auc = roc_auc_score(y_test, rf_probabilities)

print("Training Accuracy:", round(rf_train_accuracy, 3))
print("Testing Accuracy:", round(rf_accuracy, 3))
print("Recall:", round(rf_recall, 3))
print("ROC-AUC:", round(rf_auc, 3))


Training Accuracy: 1.0
Testing Accuracy: 0.747
Recall: 0.306
ROC-AUC: 0.706


In [ ]:

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

print("Top 3 Important Features:")
print(feature_importance.head(3).to_string(index=False))


Top 3 Important Features:
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739


In [ ]:

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Depth-3 Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, lr_predictions),
        depth3_test_accuracy,
        rf_accuracy
    ],
    "Recall": [
        recall_score(y_test, lr_predictions),
        recall_score(y_test, depth3_tree.predict(X_test)),
        rf_recall
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, lr_probabilities),
        roc_auc_score(
            y_test,
            depth3_tree.predict_proba(X_test)[:, 1]
        ),
        rf_auc
    ]
})

print(model_comparison.round(3).to_string(index=False))


                Model  Accuracy  Recall  ROC-AUC
  Logistic Regression     0.788   0.347    0.789
Depth-3 Decision Tree     0.778   0.224    0.700
        Random Forest     0.747   0.306    0.706


In [ ]:

from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.pipeline import Pipeline

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", round(grid_search.best_score_, 4))


Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Calculate cross-validation ROC-AUC
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tree_cv_auc = cross_val_score(
    depth3_tree, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv_auc = grid_search.best_score_

# Compare models
cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Depth-3 Decision Tree",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_cv_auc,
        tree_cv_auc,
        rf_cv_auc,
        tuned_rf_cv_auc
    ]
})

cv_results = cv_results.sort_values(
    "CV ROC-AUC",
    ascending=False
)

print(cv_results.round(4).to_string(index=False))
print("\nWinning model:", cv_results.iloc[0]["Model"])


                Model  CV ROC-AUC
  Logistic Regression      0.7737
  Tuned Random Forest      0.7673
        Random Forest      0.7466
Depth-3 Decision Tree      0.6784

Winning model: Logistic Regression


In [ ]:

winner_name = cv_results.iloc[0]["Model"]

models = {
    "Logistic Regression": lr_pipe,
    "Depth-3 Decision Tree": depth3_tree,
    "Random Forest": rf,
    "Tuned Random Forest": grid_search.best_estimator_
}

winner_model = models[winner_name]

print("Selected model:", winner_name)
print("Training CV ROC-AUC:", round(cv_results.iloc[0]["CV ROC-AUC"], 4))


Selected model: Logistic Regression
Training CV ROC-AUC: 0.7737


In [ ]:

from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    winner_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

print("Number of predictions:", len(oof_probabilities))


Number of predictions: 792


In [ ]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in [0.50, 0.35]:
    predicted_flags = (
        oof_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train, predicted_flags, zero_division=0
        ),
        "Recall": recall_score(
            y_train, predicted_flags, zero_division=0
        ),
        "F1": f1_score(
            y_train, predicted_flags, zero_division=0
        ),
        "Number Flagged": predicted_flags.sum()
    })

threshold_table = pd.DataFrame(threshold_results)

print(threshold_table.round(3).to_string(index=False))

best_threshold = threshold_table.loc[
    threshold_table["F1"].idxmax(), "Threshold"
]

print("\nChosen threshold:", best_threshold)


 Threshold  Precision  Recall    F1  Number Flagged
      0.50      0.575   0.249 0.347              73
      0.35      0.513   0.467 0.489             154

Chosen threshold: 0.35


In [ ]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score
)

# Fit the selected model on all training data
winner_model.fit(X_train, y_train)

# Predict probabilities on the test set
test_probabilities = winner_model.predict_proba(X_test)[:, 1]

# Apply the threshold chosen from training data
test_flags = (
    test_probabilities >= best_threshold
).astype(int)

# Final test metrics
final_precision = precision_score(y_test, test_flags)
final_recall = recall_score(y_test, test_flags)
final_auc = roc_auc_score(y_test, test_probabilities)
final_flagged = test_flags.sum()

print("FINAL MODEL:", winner_name)
print("SELECTED THRESHOLD:", best_threshold)

print("\nFINAL TEST RESULTS")
print("Precision:", round(final_precision, 3))
print("Recall:", round(final_recall, 3))
print("ROC-AUC:", round(final_auc, 3))
print("Number Flagged:", final_flagged)


FINAL MODEL: Logistic Regression
SELECTED THRESHOLD: 0.35

FINAL TEST RESULTS
Precision: 0.571
Recall: 0.571
ROC-AUC: 0.789
Number Flagged: 49


In [ ]:

test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[
        X_test.index, "appointment_id"
    ].values,
    "no_show": y_test.values,
    "risk_prob": test_probabilities,
    "flag": test_flags
})

test_risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

print("File saved: test_risk_scores.csv")
print("Total rows:", len(test_risk_scores))
print("Flagged appointments:",
      test_risk_scores["flag"].sum())

print("\nFirst five predictions:")
print(test_risk_scores.head())

assert len(test_risk_scores) == 198
assert test_risk_scores["flag"].sum() == final_flagged


File saved: test_risk_scores.csv
Total rows: 198
Flagged appointments: 49

First five predictions:
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0


In [ ]:

from google.colab import files

files.download("test_risk_scores.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Model Selection and Business Interpretation

The final model was selected using five-fold cross-validation on training data only, with ROC-AUC as the selection metric. The decision threshold was selected by comparing 0.50 and 0.35 using out-of-fold training predictions and choosing the higher F1 score.

Recall is important because failing to identify a genuine no-show leaves an unused doctor appointment, while an unnecessary follow-up call has a relatively small cost.

In a real clinic, the decision threshold should be chosen based on how many calls the front desk can make each evening and the financial and operational cost of empty appointment slots.

The test set was used for final evaluation only, not for model or threshold selection.

In [2]:

print("Selected model:", winner_name)
print("Threshold:", best_threshold)
print("Test precision:", round(final_precision, 3))
print("Test recall:", round(final_recall, 3))
print("Test ROC-AUC:", round(final_auc, 3))
print("Appointments flagged:", final_flagged)


NameError: name 'winner_name' is not defined